# IPR010047 클러스터 임계값 스윕 — min-seq-id 에 따라 분류가 어떻게 변하는가

2026-04 에 `--min-seq-id 0.70` 으로 한 번 돌려 219 클러스터를 얻었다. 그 값이 적절한지
판단할 근거가 없으므로, 0.10~0.90 을 0.05 간격으로 훑어 **분류가 어디서 안정하고 어디서
무너지는지** 를 본다. 보존도 분석에 쓸 서열 세트를 정하는 것이 최종 목적이다.

## 1. 설정

경로와 상수를 여기 한 곳에 모았다. **`REPO` 한 줄만 환경에 맞게 고치면 된다.**
계산 로직은 전부 `pipeline/stages/02_cluster_sweep.py` 에 있고, 이 노트북은 그것을 불러
결과를 보기만 한다. 출력으로 python/pandas 버전과 커널 위치를 확인하라.

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
import pandas as pd

REPO = Path.home() / "office_claude"                 # ← 환경이 다르면 여기만 고친다
sys.path.insert(0, str(REPO / "pipeline" / "lib"))
import nb, common
sweep = nb.load_stage("02_cluster_sweep")

MNT    = Path("/mnt/af2results/mingyu")
FASTA  = MNT / "chcodh2/protein-matching-IPR010047.fasta"
MASTER = MNT / "workspace/docking_simulation/result/homolog_split/master_1827.tsv"
OUTDIR = MNT / "workspace/docking_simulation/result/homolog_split/cluster_sweep_nb"
SWEEP, TMP = OUTDIR / "clusters", OUTDIR / "tmp"
SWEEP.mkdir(parents=True, exist_ok=True)
TMP.mkdir(parents=True, exist_ok=True)

REF_ACC    = "Q9F8A8"                                # ChCODH2
STABLE_ARI = 0.90                                    # 이 값 이상을 '안정' 으로 본다
THRESHOLDS = sweep.DEFAULT_THRESHOLDS                # 0.10~0.90, 0.05 간격 = 17개
LABELS     = sweep.make_labels(THRESHOLDS)           # 'id70' 꼴. 파일 이름이 된다
nb.env_report()

## 2. mmseqs 찾기

**인계문서 6절의 함정**: Jupyter 커널의 `PATH` 는 로그인 셸의 `PATH` 와 다르다. 터미널에서
`mmseqs` 가 되는데 여기서 안 되는 일이 흔하다. 못 찾으면 터미널에서
`readlink -f "$(which mmseqs)"` 를 돌려 절대경로를 확인하고, 아래 주석 줄을 풀어 직접 박으면 된다.
`find_mmseqs` 는 못 찾아도 예외를 던지지 않고 `None` 을 돌려주므로 셀이 터지지는 않는다.

In [ ]:
MMSEQS = nb.find_mmseqs()
# MMSEQS = "/mnt/af2results/mingyu/soft/mmseqs/bin/mmseqs"   # ← 못 찾으면 이 줄을 쓴다

THREADS = min(8, os.cpu_count() or 4)
print(f"threads : {THREADS}")

## 3. 입력 확인

돌리기 전에 입력이 기대한 것인지 **숫자를 눈으로 보는** 셀이다. FASTA 가 1,827 레코드인지,
기존 master 분할이 219 클러스터인지, 그리고 ChCODH2(Q9F8A8) 가 FASTA 안에 실제로
있는지 확인한다. 숫자가 다르면 아래 전부가 2026-04 결과와 비교 불가이므로 여기서 멈춰야 한다.

In [ ]:
universe, uni = sweep.fasta_universe(FASTA)
print(f"FASTA 레코드      {uni['n_records']}    (기대 1827)")
print(f"고유 accession    {uni['n_unique_accession']}")
print(f"중복/빈 accession {uni['n_duplicate_accession']} / {uni['n_empty_accession']}")
print(f"{REF_ACC} 포함?      {REF_ACC in universe}")

master, minfo = sweep.read_master_partition(MASTER, allow_duplicate_accession=False)
if master is None:
    print(f"\nmaster 없음 ({minfo['reason']}) — 4절의 219 비교는 건너뛴다: {MASTER}")
else:
    print(f"\nmaster 분할       {minfo['n_assignments']}개 배정 / "
          f"{minfo['n_clusters']}개 클러스터  (기대 219)")

## 4. 시험 실행 — 0.70 하나만

전체 스윕은 mmseqs 를 **17회** 돌린다. 1회에 얼마나 걸리고 tmp 를 얼마나 쓰는지 아직
아무도 모르므로, 먼저 0.70 하나만 돌려 **소요시간과 tmp 용량을 재고** 덤으로 **219 가
재현되는지** 본다. 여기서 걸린 시간 × 17 이 전체 예상 시간이다. 그 값이 감당 못 할
크기면 5절 루프를 돌리기 전에 임계값 개수를 줄여야 한다.

같이 볼 것: `n_clusters` 가 219 인가, master 와의 `ari` 가 1.0 인가. ARI 가 1.0 이면
분할은 완전히 동일하고 대표 ID 만 바뀐 것이다 (`one_to_one` 수치로 확인).

In [ ]:
T0 = 0.70
L0 = sweep.threshold_label(T0)
cmd = sweep.build_cluster_cmd(MMSEQS, FASTA, SWEEP / L0, TMP / L0, T0,
                             cov=sweep.FIXED_COV, cov_mode=sweep.FIXED_COV_MODE,
                             sensitivity=sweep.FIXED_SENSITIVITY, threads=THREADS)
print(" ".join(cmd), "\n")

t0 = time.time()
subprocess.run(cmd, check=True, capture_output=True, text=True)
elapsed = time.time() - t0
print(f"소요 {elapsed:.1f}초   tmp {sweep.dir_bytes(TMP / L0) / 2**30:.2f} GiB")
print(f"→ 전체 17회 예상 {elapsed * 17 / 60:.1f}분\n")

df0, _ = sweep.read_cluster_tsv(SWEEP / f"{L0}_cluster.tsv")
p0, _ = sweep.partition_from_cluster_df(df0, L0, allow_duplicate_accession=False)
print(f"n_clusters = {p0.nunique()}   (2026-04 결과 219)")
if master is not None:
    c = sweep.compare_partitions(p0, master)
    print(f"master 와 ARI {c['ari']:.4f}  NMI {c['nmi']:.4f}  "
          f"1:1 대응 {c['one_to_one']}개  분할 {c['n_split']}  합병 {c['n_merged']}")

## 5. 전체 스윕

나머지 임계값을 낮은 쪽부터 돌린다. 임계값마다 한 줄씩 클러스터 수와 소요시간을 찍으므로
기다리는 동안 진행 상황이 보인다. **결과 파일이 이미 있으면 mmseqs 를 건너뛰고 읽기만
하므로**, 중간에 끊겨도 이 셀을 다시 실행하면 이어진다. 0.70 은 4절에서 이미 끝냈다.

In [ ]:
partitions = {L0: p0}
runs = {L0: elapsed}

for t, label in zip(THRESHOLDS, LABELS):
    if label in partitions:
        continue
    out_tsv = SWEEP / f"{label}_cluster.tsv"
    if not out_tsv.exists():                       # 있으면 건너뛴다 → 재실행 시 이어짐
        c = sweep.build_cluster_cmd(MMSEQS, FASTA, SWEEP / label, TMP / label, t,
                                    cov=sweep.FIXED_COV, cov_mode=sweep.FIXED_COV_MODE,
                                    sensitivity=sweep.FIXED_SENSITIVITY, threads=THREADS)
        t0 = time.time()
        subprocess.run(c, check=True, capture_output=True, text=True)
        runs[label] = time.time() - t0
    d, _ = sweep.read_cluster_tsv(out_tsv)
    partitions[label], _ = sweep.partition_from_cluster_df(
        d, label, allow_duplicate_accession=False)
    took = f"{runs[label]:.1f}초" if label in runs else "기존 파일"
    print(f"{label}  seq-id {t:.2f}  {partitions[label].nunique():5d} 클러스터  {took}")

ORDER = [lb for lb in LABELS if lb in partitions]
print(f"\n완료 {len(ORDER)}/{len(LABELS)},  실행 합계 {sum(runs.values()) / 60:.1f}분")

## 6. 임계값별 지표

임계값마다 분할의 '모양' 을 한 줄로 본다. `n_clusters` 는 클러스터 수, `singleton_frac` 은
혼자인 클러스터의 비율, `largest_frac` 은 가장 큰 클러스터가 전체 서열의 몇 %를 차지하는지다.
낮은 임계값에서 `largest_frac` 이 1에 가까우면 전부 한 덩어리(해상도 없음), 높은 임계값에서
`singleton_frac` 이 1에 가까우면 아무것도 묶이지 않은 것(역시 쓸모없음)이다.

In [ ]:
metrics = pd.DataFrame([
    dict(label=lb, min_seq_id=sweep.label_to_threshold(lb),
         **common.partition_summary(partitions[lb]))
    for lb in ORDER
]).set_index("label")

metrics[["min_seq_id", "n_clusters", "n_singletons", "singleton_frac",
         "largest_cluster", "largest_frac", "median_size"]].round(3)

## 7. 인접 임계값 안정성 — 이 노트북의 핵심

인접한 두 임계값의 분할을 ARI 로 비교한다. ARI 가 높게 유지되는 구간은 **임계값을 조금
흔들어도 분류가 같다**는 뜻(안정), 급락하는 지점은 분류가 재편되는 **경계**다. 0.70 이
안정 구간의 안쪽에 있으면 그 값을 계속 쓸 근거가 되고, 경계에 걸쳐 있으면 위험하다.

**⚠ 퇴화 주의.** 전부 한 덩어리인 구간과 전부 싱글턴인 구간도 ARI 는 1.0 이 나온다.
그것은 "안정" 이 아니라 **"해상도가 쓸모없음"** 이다. 그래서 `ari_degenerate` 플래그와
6절의 `n_clusters` 를 반드시 같이 봐야 한다. 양쪽 끝(0.10 근처, 0.90 근처)이 보통 그렇다.

In [ ]:
adjacent, interp = sweep.adjacent_table(partitions, ORDER, stable_ari=STABLE_ARI,
                                        ref_acc=REF_ACC, max_list=20)
print(f"ARI 중앙값 {interp['ari_median']:.3f}  (최소 {interp['ari_min']:.3f} / "
      f"최대 {interp['ari_max']:.3f})   퇴화 쌍 {interp['n_ari_degenerate']}개")

print(f"\n■ 안정 구간 (인접 ARI >= {STABLE_ARI} 가 연속되는 범위)")
for iv in interp["stable_intervals"]:
    print(f"   {iv['low']:.2f} ~ {iv['high']:.2f}   {iv['n_steps']}스텝   "
          f"최소 ARI {iv['min_ari']:.3f}")

print("\n■ 경계 (ARI 낮은 쪽부터 5개)")
for b in interp["boundaries"][:5]:
    print(f"   {b['low']:.2f} → {b['high']:.2f}   ARI {b['ari']:.3f}   "
          f"합병된 클러스터 {b['n_coarser_merged']}개")

print(f"\n0.70 이 속한 안정 구간 : {interp['reference_in_stable_interval']}")
print(f"0.70 이 그 구간의 끝점? : {interp['reference_in_stable_interval_at_edge']}")

adjacent[["label_low", "label_high", "ari", "nmi", "n_clusters_coarser",
          "n_clusters_finer", "is_stable", "ari_degenerate"]].round(3)

## 8. ChCODH2(Q9F8A8) 클러스터 추적

각 임계값에서 Q9F8A8 이 속한 클러스터의 크기와 대표 ID 를 따라간다. 임계값을 내릴 때
다른 클러스터와 합쳐지는 지점이 보이는데, **보존도 분석 대상 세트를 정하는 근거가 이 표다.**
크기가 너무 작으면 보존도를 계산할 서열이 없고, 갑자기 커지는 지점 아래로는 다른 CODH
계열이 섞여 들어온 것이다. `ref_is_mm_rep` 는 Q9F8A8 자신이 대표로 뽑혔는지 여부다.

In [ ]:
ref_df, ref_per_label, ref_info = sweep.track_reference(
    partitions, ORDER, REF_ACC, max_members=50)
print(f"{REF_ACC} 발견 {ref_info['n_thresholds_found']}/{len(ORDER)} 임계값,  "
      f"클러스터 크기 {ref_info['size_min']} ~ {ref_info['size_max']}")

print("\n■ 임계값을 내릴 때 Q9F8A8 클러스터가 합쳐지는 지점")
for s in interp["ref_merge_steps"]:
    print(f"   {s['high']:.2f} → {s['low']:.2f}   출처 클러스터 "
          f"{s['n_source_clusters']}개, +{s['n_added']}서열")

ref_df[["min_seq_id", "ref_found", "ref_cluster", "ref_cluster_size", "ref_is_mm_rep"]]

## 9. 대조군 추적 (선택)

`pipeline/lib/markers.txt` 에 적은 대조군이 어느 임계값에서 서로 갈라지는지 본다.
**지금 markers.txt 에는 Q9F8A8 하나뿐이라 쌍 비교(`n_pairs`)가 공집합이고, 이 셀은
사실상 8절의 반복이다.** ChCODH1/3/4/5·RrCODH·DvCODH·MtCODH 의 UniProt accession 을
markers.txt 왼쪽 칼럼에 채우면(주석 해제) 그때부터 쌍 비교가 의미를 가진다.

In [ ]:
markers, mk = sweep.read_markers(REPO / "pipeline" / "lib" / "markers.txt")
print(f"markers.txt : {mk['n_markers']}개 — {[m['marker_accession'] for m in markers]}")

mk_df, mk_per_label, mk_info = sweep.track_markers(partitions, ORDER, markers, max_list=20)
print(f"쌍 비교     : {mk_info['n_pairs']}개   (accession 을 2개 이상 채워야 쌍이 생긴다)")
print(f"FASTA 에 없는 마커 : {mk_info['missing_markers']}")
for p in mk_info["pairs"]:
    print(f"   {p['label_a']} vs {p['label_b']}  같은 클러스터인 최고 임계값 "
          f"{p['highest_together']}")

mk_df[mk_df["found"]][["label", "marker_label", "cluster", "cluster_size",
                       "n_co_markers"]].tail(6)

## 10. 그림 (선택)

matplotlib 이 있으면 두 장을 그린다. 왼쪽은 임계값 대 클러스터 수(로그 y축), 오른쪽은
인접 ARI 다. 빨간 점선이 0.70, 회색 점선이 안정 기준선이다. matplotlib 이 없으면
안내만 찍고 넘어간다 — 6·7절 표로 판단은 충분히 된다.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib 이 없다. 6·7절 표로 충분하다. (설치하지 않고 넘어가도 된다)")

if plt is not None:
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(metrics["min_seq_id"], metrics["n_clusters"], "o-")
    ax[0].set_yscale("log")
    ax[0].set_xlabel("min-seq-id"); ax[0].set_ylabel("클러스터 수 (log)")
    ax[0].axvline(0.70, ls="--", c="red")
    ax[1].plot(adjacent["min_seq_id_high"], adjacent["ari"], "o-")
    ax[1].axhline(STABLE_ARI, ls="--", c="gray")
    ax[1].axvline(0.70, ls="--", c="red")
    ax[1].set_xlabel("min-seq-id (인접 쌍의 높은 쪽)"); ax[1].set_ylabel("인접 ARI")
    ax[1].set_ylim(0, 1.05)
    fig.tight_layout()
    plt.show()

## 11. 저장

표 네 개를 tsv 로, 7절 해석을 json 으로, 재현 정보(입력 해시·명령·버전)를
`common.write_params` 로 `params.json` 에 남긴다. 어떤 파일이 어디 생겼는지 출력하므로
다음에 이어 쓸 때 그 경로를 보면 된다. 기존 `master_1827.tsv` 는 읽기만 했고 건드리지 않았다.

In [ ]:
metrics.to_csv(OUTDIR / "sweep_metrics.tsv", sep="\t")
adjacent.to_csv(OUTDIR / "sweep_adjacent.tsv", sep="\t", index=False)
ref_df.to_csv(OUTDIR / "sweep_chcodh2.tsv", sep="\t", index=False)
mk_df.to_csv(OUTDIR / "sweep_markers.tsv", sep="\t", index=False)
(OUTDIR / "interpretation.json").write_text(
    json.dumps(interp, indent=2, ensure_ascii=False, default=str))

common.write_params(
    OUTDIR, "02_cluster_sweep (notebook)",
    inputs=[FASTA, MASTER],
    outputs=sorted(OUTDIR.glob("sweep_*.tsv")),
    commands=[" ".join(cmd)],
    mmseqs=MMSEQS, thresholds=THRESHOLDS, stable_ari=STABLE_ARI,
    fixed=dict(c=sweep.FIXED_COV, cov_mode=sweep.FIXED_COV_MODE,
               s=sweep.FIXED_SENSITIVITY, threads=THREADS),
    n_clusters_by_label={lb: int(partitions[lb].nunique()) for lb in ORDER},
)

for f in sorted(OUTDIR.glob("*")):
    print(f)